Part A Overflow Test

In [ ]:
import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge, Timer


@cocotb.test()
async def test_exact_overflow_boundary(dut):
    """
    Test the exact overflow boundary.
    Calculate exact values to hit 2^31-1 and then overflow by 1.
    """
    
    clock = cocotb.start_soon(Clock(dut.clk, 10, units="ns").start())
    
    # Reset
    dut.rst.value = 1
    dut.a.value = 0
    dut.b.value = 0
    await RisingEdge(dut.clk)
    dut.rst.value = 0
    await RisingEdge(dut.clk)
    
    dut._log.info("Testing exact overflow boundary")
    
    MAX_INT32 = 2**31 - 1  # 2,147,483,647
    
    # Build up to MAX_INT32 - 100
    target = MAX_INT32 - 100
    product = 127 * 127  # 16,129
    cycles = target // product
    
    dut.a.value = 127
    dut.b.value = 127
    
    for i in range(cycles):
        await RisingEdge(dut.clk)
    
    await Timer(1, units="ns")
    current = int(dut.out.value.signed_integer)
    dut._log.info(f"Current value: {current:,}")
    dut._log.info(f"Max value:     {MAX_INT32:,}")
    dut._log.info(f"Remaining:     {MAX_INT32 - current:,}")
    
    # Now add exactly enough to reach MAX_INT32 - 1
    remaining = MAX_INT32 - current
    
    # Use a=10, b=10 for fine control (product = 100)
    dut.a.value = 10
    dut.b.value = 10
    
    small_cycles = (remaining - 50) // 100  # Leave 50 margin
    
    dut._log.info(f"\nAdding 10*10=100 for {small_cycles} cycles")
    
    for i in range(small_cycles):
        await RisingEdge(dut.clk)
    
    await Timer(1, units="ns")
    current = int(dut.out.value.signed_integer)
    remaining = MAX_INT32 - current
    dut._log.info(f"Value now:  {current:,}")
    dut._log.info(f"Remaining:  {remaining:,}")
    
    # Add 1 more cycle
    dut._log.info(f"\nAdding one more 10*10=100...")
    await RisingEdge(dut.clk)
    await Timer(1, units="ns")
    current = int(dut.out.value.signed_integer)
    dut._log.info(f"Result: {current:,}")
    
    if current < 0:
        dut._log.warning("⚠️ OVERFLOWED to negative!")
    
    # Keep adding to see the wrap
    dut._log.info("\nContinuing to add 10*10=100 to observe wrap behavior...")
    for i in range(5):
        await RisingEdge(dut.clk)
        await Timer(1, units="ns")
        current = int(dut.out.value.signed_integer)
        dut._log.info(f"  Cycle {i+1}: {current:,}")

# MAC Module Overflow Test Results

## Test: Exact Overflow Boundary

Testing what happens when the 32-bit signed accumulator approaches and exceeds 2³¹-1.

---

## Test Configuration

- **Test Module**: `test_overflow_precise`
- **Test Function**: `test_exact_overflow_boundary`
- **Simulator**: Icarus Verilog 13.0 (stable)
- **Framework**: cocotb v2.0.1
- **Total Simulation Time**: 1,331,911.00 ns (~1.33 ms)
- **Real Time**: 10.41 seconds

---

## Test Sequence

### Phase 1: Build to Near Maximum
- Used **127 × 127 = 16,129** per cycle
- Reached: **2,147,479,576**
- Distance from max: **4,071**

### Phase 2: Fine Control Approach
- Used **10 × 10 = 100** per cycle
- Ran for **40 cycles**
- Reached: **2,147,483,576**
- Distance from max: **71**

### Phase 3: Cross the Boundary
- Added one more **10 × 10 = 100**
- **Result: -2,147,483,620** ⚠️
- **OVERFLOW DETECTED!**

### Phase 4: Observe Wraparound
Continued adding 10 × 10 = 100 for 5 more cycles:

| Cycle | Accumulator Value |
|-------|-------------------|
| 1 | -2,147,483,520 |
| 2 | -2,147,483,420 |
| 3 | -2,147,483,320 |
| 4 | -2,147,483,220 |
| 5 | -2,147,483,120 |

---

## Key Observations

### The Math Behind the Overflow

**32-bit Signed Integer Range:**
- Minimum: -2,147,483,648 (−2³¹)
- Maximum: +2,147,483,647 (+2³¹−1)

**What Happened:**
```
Before overflow:  2,147,483,576
Add 10 × 10:            + 100
                 ─────────────
Expected (64-bit): 2,147,483,676
Actual (32-bit):  -2,147,483,620  ← Two's complement wraparound
```

**The Wraparound Calculation:**
```python
# When exceeding 2^31-1, the value wraps using two's complement
expected = 2_147_483_676
wrapped = expected - 2**32  # Subtract 2^32 to get signed value
wrapped = -2_147_483_620    # ✓ Matches actual result!
```

---

## Visual Representation

```
                    2³¹-1 = 2,147,483,647
                           ↓
    ┌──────────────────────┼──────────────────────┐
    │                      │                      │
Positive                   │                  Negative
    │                      ↓                      │
    │              Overflow happens here!         │
    │                      ↓                      │
    │         Value: 2,147,483,576                │
    │         Add:              100               │
    │         Result wraps to: -2,147,483,620     │
    │                      ↓                      │
    └──────────────────────────────────────────────┘
                           │
                    -2³¹ = -2,147,483,648
```

---

## Test Results Summary

✅ **Test PASSED** - Overflow behavior correctly demonstrated

| Metric | Value |
|--------|-------|
| Test Status | **PASS** |
| Simulation Time | 1,331,911 ns |
| Real Time | 10.41 s |
| Throughput Ratio | 127,976.61 ns/s |

---

## Conclusions

### ✅ Verified Behavior

1. **Two's Complement Overflow**: When accumulator exceeds 2³¹−1, it wraps to negative values
2. **Predictable Wraparound**: The overflow follows standard two's complement arithmetic
3. **Continuous Operation**: After overflow, accumulation continues with wrapped negative values

### 🔍 Implications for Hardware Design

**This overflow behavior is:**
- ✓ **Expected** - Standard for signed 32-bit integers
- ✓ **Deterministic** - Same result every time
- ⚠️ **Potentially Problematic** - If unchecked, can lead to incorrect results

### 💡 Design Recommendations

For production MAC modules, consider adding:

1. **Saturation Logic**
   ```systemverilog
   if (overflow_detected)
       out <= 32'sd2147483647;  // Clamp to max
   ```

2. **Overflow Flag**
   ```systemverilog
   output logic overflow;
   assign overflow = (out[31] != product[31]) && (product[31] == accumulator[31]);
   ```

3. **Wider Accumulator**
   ```systemverilog
   output logic signed [63:0] out;  // 64-bit accumulator
   ```

---

## Code Reference

**MAC Module (Current Implementation):**
```systemverilog
module mac (
    input  logic        clk,
    input  logic        rst,
    input  logic signed [7:0]  a,
    input  logic signed [7:0]  b,
    output logic signed [31:0] out
);
    always_ff @(posedge clk) begin
        if (rst)
            out <= 32'sd0;
        else
            out <= out + 32'(a * b);  // No overflow protection!
    end
endmodule
```

---

## Deprecation Warnings (Fixed for Future Use)

The test generated several deprecation warnings. Here are the fixes:

| Old Syntax | New Syntax |
|------------|------------|
| `units="ns"` | `unit="ns"` |
| `.signed_integer` | `.to_signed()` |

**Updated code snippets:**
```python
# Old: await Timer(1, units="ns")
await Timer(1, unit="ns")

# Old: current = int(dut.out.value.signed_integer)
current = int(dut.out.value.to_signed())
```
```